# WHACS offshore waves at the goals

Reads WHACS (bulk and the four partitions: 0 = wind sea, 1-3 = swells) at the
WHACS node of every goal from the local copy of the global archive, cleans the
peak periods and writes the processed zarr used by the dynamic runs and the
reconstruction. The partitions are written as WHACS gives them: the energy they
miss is measured, and closed on the bulk, by the calibration (07b).

The archive is the `archive` of the WHACS `offshore_waves` entry: one folder per
variable with monthly files on the global SMC mesh, as CSIRO publishes them
([doi:10.25919/shdk-7p29](https://doi.org/10.25919/shdk-7p29)). Only the file
chunks that hold the goal nodes are read (`utils.whacs`).

**The archive is not part of HyGloW.** On the GeoOcean cluster it is
`/lustre/geocean/DATA/CSIRO/WHACS/global-smc`. Elsewhere, download the 20
variables read here (`hs`, `fp`, `dir`, `spr` and `phs`, `ptp`, `pdir`, `pspr`
0-3; about 14 TB for 1979-2026) or ask GeoOcean for the goal series of your case.

**Prerequisites:** `04_Select_Goals_by_Hindcasts.ipynb` (`<Case>_Goals.json`,
with a `whacs` node per goal).
**Next:** `07b_Satellite_Calibration.ipynb` validates (and calibrates) this
hindcast against satellite altimetry.

## Configuration


In [ ]:
import xarray as xr

from utils import whacs
from utils.config import get_offshore_wave, resolve

offshore_name = "WHACS"  # offshore_waves entry with the archive
period = None  # months to read, e.g. slice("1979", "2023"); None = the whole archive
workers = 16  # processes reading the monthly files

archive = whacs.archive_path(offshore_name)
print(f"archive: {archive}")
print(f"output : {resolve(get_offshore_wave(offshore_name)['data_path'])}")

## Goal nodes

The archive seapoint of each goal's WHACS node (goals JSON `hindcast.whacs`);
`distance` (degrees) is ~0 when the node is a WHACS seapoint.

In [ ]:
whacs.goal_seapoints(archive, offshore_name)

## Read the archive

Every variable of every month in `period` at the goal seapoints (`tp = 1 / fp`).

In [ ]:
whacs_data = whacs.whacs_at_goals(offshore_name, period=period, workers=workers)
whacs_data

Bulk `tp = 1 / fp` explodes when the spectral peak falls below the WHACS
frequency grid (hours with Tp of 50-250 s). Those take the peak period of the
most energetic partition (`clean_peak_periods`); partition periods above
`max_tp` become NaN.

In [ ]:
from bluemath_tk.waves.hywaves.offshore import clean_peak_periods

max_tp = 25.0  # s, largest valid peak period
whacs_data = clean_peak_periods(whacs_data, max_tp=max_tp)
print(f"bulk tp replaced: {whacs_data.attrs['tp_replaced']}, partition tp removed: {whacs_data.attrs['ptp_removed']}")

## Write the zarr

In [ ]:
whacs_data.to_zarr(resolve(get_offshore_wave(offshore_name)["data_path"]), mode="w")

In [ ]:
xr.open_zarr(resolve(get_offshore_wave(offshore_name)["data_path"])).load()